# BM25 Ranking (Better for Short Queries)
BM25 is often more effective than TF-IDF in search tasks, especially when queries are short and precise.

In [1]:
import sys
import os
import json 
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

ENV = 'dev'

# Load config
with open("config.json") as f:
    config = json.load(f)

if ENV == 'dev':
    base_path = config[f"{ENV}_path"]  
    data_path = os.path.join(base_path, "data")
    model_path = os.path.join(base_path, "models")
    print("Base path:", base_path)    
    print("Data path:", data_path)
    print("Model path:", model_path)

/Users/jillchow/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


Base path: /Users/jillchow/HBS/hbs_search_engine
Data path: /Users/jillchow/HBS/hbs_search_engine/data
Model path: /Users/jillchow/HBS/hbs_search_engine/models


In [2]:
queryfile_name = "query.csv" 
queryfile_path = os.path.join(data_path, queryfile_name)
productfile_name = "product.csv" 
productfile_path = os.path.join(data_path, productfile_name)
labelfile_name = "label.csv" 
labelfile_path = os.path.join(data_path, labelfile_name)


query_df = pd.read_csv(queryfile_path, sep='\t')
product_df = pd.read_csv(productfile_path, sep='\t')
label_df = pd.read_csv(labelfile_path, sep='\t')

print('query_df: search queries')
display(query_df.head()) # watch for null values in query class column 
query_df.info()

print('\n product_df: product information')
display(product_df.head())
product_df.info() # watch for null values other than product_id, product_name, product_features

print('\n label_df: ground truth labels')
display(label_df.head())
label_df.info()

query_df: search queries


,query_id,query,query_class
0,0,salon chair,Massage Chairs
1,1,smart coffee table,Coffee & Cocktail Tables
2,2,dinosaur,Kids Wall Décor
3,3,turquoise pillows,Accent Pillows
4,4,chair and a half recliner,Recliners


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 480 entries, 0 to 479
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   query_id     480 non-null    int64 
 1   query        480 non-null    object
 2   query_class  474 non-null    object
dtypes: int64(1), object(2)
memory usage: 11.4+ KB

 product_df: product information


,product_id,product_name,product_class,category hierarchy,product_description,product_features,rating_count,average_rating,review_count
0,0,solid wood platform bed,Beds,Furniture / Bedroom Furniture / Beds & Headboa...,"good , deep sleep can be quite difficult to ha...",overallwidth-sidetoside:64.7|dsprimaryproducts...,15.0,4.5,15.0
1,1,all-clad 7 qt . slow cooker,Slow Cookers,Kitchen & Tabletop / Small Kitchen Appliances ...,"create delicious slow-cooked meals , from tend...",capacityquarts:7|producttype : slow cooker|pro...,100.0,2.0,98.0
2,2,all-clad electrics 6.5 qt . slow cooker,Slow Cookers,Kitchen & Tabletop / Small Kitchen Appliances ...,prepare home-cooked meals on any schedule with...,features : keep warm setting|capacityquarts:6....,208.0,3.0,181.0
3,3,all-clad all professional tools pizza cutter,"Slicers, Peelers And Graters",Browse By Brand / All-Clad,this original stainless tool was designed to c...,overallwidth-sidetoside:3.5|warrantylength : l...,69.0,4.5,42.0
4,4,baldwin prestige alcott passage knob with roun...,Door Knobs,Home Improvement / Doors & Door Hardware / Doo...,the hardware has a rich heritage of delivering...,compatibledoorthickness:1.375 '' |countryofori...,70.0,5.0,42.0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42994 entries, 0 to 42993
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   product_id           42994 non-null  int64  
 1   product_name         42994 non-null  object 
 2   product_class        40142 non-null  object 
 3   category hierarchy   41438 non-null  object 
 4   product_description  36986 non-null  object 
 5   product_features     42994 non-null  object 
 6   rating_count         33542 non-null  float64
 7   average_rating       33542 non-null  float64
 8   review_count         33542 non-null  float64
dtypes: float64(3), int64(1), object(5)
memory usage: 3.0+ MB

 label_df: ground truth labels


,id,query_id,product_id,label
0,0,0,25434,Exact
1,1,0,12088,Irrelevant
2,2,0,42931,Exact
3,3,0,2636,Exact
4,4,0,42923,Exact


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 233448 entries, 0 to 233447
Data columns (total 4 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   id          233448 non-null  int64 
 1   query_id    233448 non-null  int64 
 2   product_id  233448 non-null  int64 
 3   label       233448 non-null  object
dtypes: int64(3), object(1)
memory usage: 7.1+ MB


In [ ]:
# %pip install rank_bm25

Note: you may need to restart the kernel to use updated packages.


In [21]:
from rank_bm25 import BM25Okapi
import nltk
from nltk.tokenize import word_tokenize

# nltk.download('punkt')

corpus = (product_df['product_name'] + ' ' + product_df['product_description']).fillna("").astype(str).tolist()
tokenized_corpus = [word_tokenize(doc.lower()) for doc in corpus]
bm25 = BM25Okapi(tokenized_corpus)


def get_top_products_bm25(query, top_n=10):
    tokenized_query = word_tokenize(query.lower())
    scores = bm25.get_scores(tokenized_query)
    top_indices = np.argsort(scores)[-top_n:][::-1]
    return top_indices

In [24]:
def get_top_product_ids_for_query(query, product_df):
    top_product_indices = get_top_products_bm25(query, top_n=10)
    top_product_ids = product_df.iloc[top_product_indices]['product_id'].tolist()
    return top_product_ids

# update to use the label_df, and add to the parameters
def get_exact_matches_for_query(query_id, label_df):
    grouped_label_df = label_df.groupby('query_id')
    query_group = grouped_label_df.get_group(query_id)
    exact_matches = query_group.loc[query_group['label'] == 'Exact']['product_id'].values
    return exact_matches


In [23]:
query_df['top_product_ids'] = query_df['query'].apply(
    lambda q: get_top_product_ids_for_query(q, product_df)
)

In [25]:
# adding the list of exact match product_IDs from labels_df
query_df['relevant_ids'] = query_df['query_id'].apply(
      lambda qid: get_exact_matches_for_query(qid, label_df)
)

In [26]:
import importlib
import helper
importlib.reload(helper)
from helper import calculate_tfidf, get_top_products, map_at_k, get_top_product_ids_for_query, get_exact_matches_for_query

In [27]:
# now assign the map@k score
query_df['map@k'] = query_df.apply(lambda x: map_at_k(x['relevant_ids'], x['top_product_ids'], k=10), axis=1)

query_df.loc[:, 'map@k'].mean()

0.3443627094356261

In [ ]:
# lets add another clean data step before applying the bm25, see if it helps

import re
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

# nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

def clean_and_tokenize(text):
    # Lowercase
    text = text.lower()
    # Remove special characters and digits
    text = re.sub(r'[^a-z\s]', '', text)
    # Tokenize
    tokens = word_tokenize(text)
    # Remove stopwords
    tokens = [t for t in tokens if t not in stop_words]
    return tokens


In [ ]:
# Rebuild BM25 with cleaned product text
# it didn't help:
# text may be pretty clean already, and 
# BM25 isn't sensitive to Punctuation or Case by Default or stop words
# Stopword Removal Can Hurt in E-commerce: Example: “Table for Kids” ≠ “Table”

corpus_cleaned = (product_df['product_name'] + ' ' + product_df['product_description']).fillna("").astype(str)
tokenized_corpus = corpus_cleaned.apply(clean_and_tokenize).tolist()

bm25 = BM25Okapi(tokenized_corpus)

def get_top_products_bm25_cleaned(query, top_n=10):
    tokenized_query = clean_and_tokenize(query)
    scores = bm25.get_scores(tokenized_query)
    top_indices = np.argsort(scores)[-top_n:][::-1]
    return top_indices

def get_top_product_ids_for_query(query, product_df):
    top_product_indices = get_top_products_bm25_cleaned(query, top_n=10)
    top_product_ids = product_df.iloc[top_product_indices]['product_id'].tolist()
    return top_product_ids

# update to use the label_df, and add to the parameters
def get_exact_matches_for_query(query_id, label_df):
    grouped_label_df = label_df.groupby('query_id')
    query_group = grouped_label_df.get_group(query_id)
    exact_matches = query_group.loc[query_group['label'] == 'Exact']['product_id'].values
    return exact_matches


In [36]:
query_df['top_product_ids'] = query_df['query'].apply(
    lambda q: get_top_product_ids_for_query(q, product_df)
)

# adding the list of exact match product_IDs from labels_df
query_df['relevant_ids'] = query_df['query_id'].apply(
      lambda qid: get_exact_matches_for_query(qid, label_df)
)

# now assign the map@k score
query_df['map@k'] = query_df.apply(lambda x: map_at_k(x['relevant_ids'], x['top_product_ids'], k=10), axis=1)

query_df.loc[:, 'map@k'].mean()

0.3401150793650794

In [37]:
query_df['top_product_ids'] = query_df['query'].apply(lambda q: product_df.iloc[get_top_products_bm25_cleaned(q)].product_id.tolist())
query_df['map@k'] = query_df.apply(lambda x: map_at_k(x['relevant_ids'], x['top_product_ids'], k=10), axis=1)
print("🧼 Cleaned BM25 MAP@10:", query_df['map@k'].mean())


🧼 Cleaned BM25 MAP@10: 0.3401150793650794
